# Phase 4: Model Evaluation (The 'Link Hide' Strategy)

Unlike standard machine learning where we split independent rows of data, **Link Prediction** requires maintaining the structural integrity of the network. We use a technique called **Edge Hiding** (or Temporal/Structural Splitting).

## Step 1: The Train-Test Split

In this approach, we treat the recommendation problem as 'completing' a partially observed graph:
1.  **Training Set (80%)**: We build our `G_train` using 80% of the 'positive' interactions (ratings $\ge 4$). The model only 'sees' these connections.
2.  **Test Set (20%)**: We 'hide' the remaining 20% of edges. These represent the 'Ground Truth'—the actual preferences we want the model to recover.

**Why positive links?** We only hide edges where the user liked the movie (rating $\ge 4$), because recommending a movie the user would dislike is considered a failure in a real recommender system.

In [6]:
import pandas as pd
import networkx as nx
import random
from sklearn.model_selection import train_test_split
from collections import Counter

# 1. Load and prefix data
df = pd.read_csv('data/u.data', sep='\t', names=['u_id', 'm_id', 'rating', 'ts'])
pos_df = df[df['rating'] >= 4].copy()
pos_df['u'] = 'u_' + pos_df['u_id'].astype(str)
pos_df['m'] = 'm_' + pos_df['m_id'].astype(str)

# 2. Split edges into Train (80%) and Test (20%)
train_edges, test_edges = train_test_split(pos_df, test_size=0.2, random_state=42)

print(f"Total Positive Edges: {len(pos_df)}")
print(f"Training Edges (to build graph): {len(train_edges)}")
print(f"Test Edges (to predict): {len(test_edges)}")

# 3. Build the Training Graph
G_train = nx.Graph()
G_train.add_edges_from(zip(train_edges['u'], train_edges['m']))

print(f"\nTraining Graph built with {G_train.number_of_nodes()} nodes.")

Total Positive Edges: 55375
Training Edges (to build graph): 44300
Test Edges (to predict): 11075

Training Graph built with 2346 nodes.


## Step 2: Evaluating with Precision & Recall @ K

In a recommender system, users rarely look past the first few results. Therefore, we evaluate the **Top-K** performance.

### 1. Precision @ K
**Precision** measures **Quality**: "Of the K items I suggested, how many were actually relevant?"
> Formula: $Precision@K = \frac{|Recommended \cap Hidden|}{|Recommended|}$

### 2. Recall @ K
**Recall** measures **Coverage**: "Of all the items the user liked (in the test set), how many did I manage to find in my top K?"
> Formula: $Recall@K = \frac{|Recommended \cap Hidden|}{|Hidden|}$

**The Trade-off:** Precision tells you if your recommendations are 'noisy' (full of irrelevant items), while Recall tells you if you are 'missing' items the user would have actually enjoyed.

In [7]:
# Re-defining our baseline recommender functions for the evaluation notebook

def get_jaccard_similarity(user1, user2, graph):
    try:
        movies1 = set(graph.neighbors(user1))
        movies2 = set(graph.neighbors(user2))
        intersection = len(movies1.intersection(movies2))
        union = len(movies1.union(movies2))
        return intersection / union if union > 0 else 0.0
    except:
        return 0.0

def get_top_k_recommendations(target_user, graph, k=5):
    all_users = [n for n in graph.nodes if n.startswith('u_')]
    
    # Find peers (limit search to 200 users for performance during evaluation)
    peer_pool = random.sample(all_users, min(200, len(all_users)))
    sims = []
    for other_user in peer_pool:
        if other_user == target_user: continue
        sims.append((other_user, get_jaccard_similarity(target_user, other_user, graph)))
    sims.sort(key=lambda x: x[1], reverse=True)
    top_peers = sims[:5]
    
    # Get candidate movies
    try:
        seen_movies = set(graph.neighbors(target_user))
    except:
        seen_movies = set()
        
    candidates = []
    for peer, _ in top_peers:
        try:
            candidates.extend(list(set(graph.neighbors(peer)) - seen_movies))
        except:
            continue
    
    # Rank by frequency
    recommendations = [m for m, count in Counter(candidates).most_common(k)]
    return recommendations

def evaluate_user(target_user, training_graph, test_edges_df, k=5):
    hidden_movies = set(test_edges_df[test_edges_df['u'] == target_user]['m'])
    if not hidden_movies: return None
    
    recommendations = get_top_k_recommendations(target_user, training_graph, k=k)
    hits = len(set(recommendations).intersection(hidden_movies))
    return hits / k

## Step 3: Global Evaluation

Finally, we calculate the **Mean Precision @ 5** across a sample of 50 users. This gives us the overall health of our recommender.

In [8]:
import numpy as np

# 1. Get users who have at least one hidden link in the test set
test_users = test_edges['u'].unique().tolist()
sample_users = random.sample(test_users, 50)

precisions = []
print("Evaluating...")

for u in sample_users:
    p = evaluate_user(u, G_train, test_edges, k=5)
    if p is not None: precisions.append(p)

mean_p = np.mean(precisions)
print(f"\nEvaluation finished!")
print(f"Mean Precision @ 5: {mean_p*100:.2f}%")
print(f"This means that on average, our model correctly predicted {mean_p*5:.2f} out of 5 hidden movies per user.")

Evaluating...

Evaluation finished!
Mean Precision @ 5: 16.40%
This means that on average, our model correctly predicted 0.82 out of 5 hidden movies per user.
